---
format:
  html:
    theme:
      light: cosmo
      dark: darkly
    toc: true
    number-sections: true
    code-fold: true
    code-summary: "Show code"
    embed-resources: true
---

## Initialization 

In [ ]:
# Core
import numpy as np
import polars as pl

# data loading
import pyreadr

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Stats / ML
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.manifold import MDS

from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.stats import spearmanr

# Reproducibility
np.random.seed(42)


### Data download

In [ ]:
import os

os.makedirs("data", exist_ok=True)

!wget -nc -P data https://www.mimuw.edu.pl/~noble/courses/MultivariateStatistics/data/pendigits.txt
!wget -nc -P data https://www.mimuw.edu.pl/~noble/courses/MultivariateStatistics/data/ozone.csv
!wget -nc -P data https://www.mimuw.edu.pl/~noble/courses/MultivariateStatistics/data/primate.scapulae.txt
!wget -nc -P data https://www.mimuw.edu.pl/~noble/courses/MultivariateStatistics/data/color.stimuli.rda
!wget -nc -P data https://www.mimuw.edu.pl/~noble/courses/MultivariateStatistics/data/wdbc.rda


## Analysis

### Exercise 1

0. Load data

In [ ]:
# load data
pendigits = pl.read_csv(
    "data/pendigits.txt",
    has_header=False,
    separator=" "
)
pendigits

X = pendigits[:, :16]
y = pendigits[:, 17]

X, y


1. Compute variances and draw corr plot to see relation

In [ ]:
variances = X.var()
print(variances)
# pendigits
# differance is actually pretty high similiarty ? cor
corr = X.corr()

threshold = 0.3
mask = np.abs(corr) < threshold

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr,
    mask=mask,
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    cbar_kws={"label": "Correlation"}
)
plt.title(f"Correlations | |r| ≥ {threshold}")
plt.show()

2. 

About methodology

In sklearn.decomposition module PCA is automatically center data but do not normalize it. Performing PCA on dataframe `X` is equivalent to making it on covariance matrix. 

3.

Q: How many PCs explain 80% resp. 90% of the total variation in the data?

A: Respectively 5 and 7 PCs.

In [ ]:
pca_cov = PCA()
pca_cov.fit(X)

explained_var = pca_cov.explained_variance_ratio_
cum_explained_var = np.cumsum(explained_var)

# Results
pca_table = pl.DataFrame({
    "n_PC": np.arange(1, len(cum_explained_var) + 1),
    "cumulative_variance_explained": cum_explained_var
})

pca_table.transpose(include_header = True)

4. 

Q: Display the first three PCs using pairwise scatterplots.

A:

We see that more often we need to observe negative values. That is because `PC1` have more density located on interval $[-100, 0)$.

We also see that there are no outliers. That is because on scatter plots for `PCi` and `PCj` where $i \not= j$ are centered and look similiar to two dimensional Gaussian distribution with only diagonal values. 

In [ ]:
Z = pca_cov.transform(X_np)[:, :3]

df_pca = pl.DataFrame({
    "PC1": Z[:, 0],
    "PC2": Z[:, 1],
    "PC3": Z[:, 2],
})

sns.pairplot(df_pca.to_pandas())
plt.show()


5. 

Here we are carrying out PCA using correlation matrix. To do so we need first transform X matrix with `StandardScaler` which centers data and make normalize it by making $\mathrm{Var}(X) = 1$. Because of that normalization covariance matrix used during PCA is synonymous to correlation matrix. 

In [ ]:
X_scaled = StandardScaler().fit_transform(X)

pca_corr = PCA()
pca_corr.fit(X_scaled)

explained_corr = pca_corr.explained_variance_ratio_


6

I extended task by also adding two addiotnal plots
- Variance graph which explains how much variance is explained by next enumeration
- Condition number graph, which shows how much more first component explains (eigenvalue of PC1) compared to $i$ component (eigenvalue PCi)


7.

We see that after 8 / 9 components we observe rapidly growth of condition number as well as plateau effect on Variance graph. 

I would state that after those 8 / 8 components there is ill-conditioning

In [ ]:
pca_corr.explained_variance_

In [ ]:
plt.figure(figsize=(8, 5))

eigvals_cov = pca_cov.explained_variance_
eigvals_corr = pca_corr.explained_variance_

plt.plot(
    eigvals_cov,
    marker="o",
    color="blue",
    label="PCA (covariance)"
)

plt.plot(
    eigvals_corr,
    marker="o",
    color="red",
    label="PCA (correlation)"
)

plt.xlabel("Number of principal components")
plt.ylabel("Eigenvalue")
plt.title("Scree plot: eigenvalues (covariance vs correlation)")
plt.legend()
plt.grid(True)

plt.show()


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    np.cumsum(explained_var),
    marker="o",
    color="blue",
    label="PCA (covariance)"
)

plt.plot(
    np.cumsum(explained_corr),
    marker="o",
    color="red",
    label="PCA (correlation)"
)

plt.xlabel("Number of principal components")
plt.ylabel("Cumulative explained variance")
plt.title("Varience graph: covariance vs correlation")
plt.legend()
plt.grid(True)

plt.show()


In [ ]:
# Condition numbers relative to the first eigenvalue
cond_cov = eigvals_cov[0] / eigvals_cov
cond_corr = eigvals_corr[0] / eigvals_corr

plt.figure(figsize=(8, 5))

plt.plot(
    cond_cov,
    marker="o",
    color="blue",
    label="PCA (covariance)"
)

plt.plot(
    cond_corr,
    marker="o",
    color="red",
    label="PCA (correlation)"
)

plt.xlabel("Principal component index")
plt.ylabel("Condition number (λ₁ / λᵢ)")
plt.title("Condition number growth across principal components")
# plt.yscale("log")
plt.legend()
plt.grid(True, which="both")

plt.show()


### Exercise 2

**Methodology**

Mantel test, consider if components in two matrixes are significantly associated with each other (pairwise). In our case we have two matrices. One describe ozon level difference between two stations (Av8top) second describe distance between those two stations (euclidean).

This is permutation test. (we permutate matrix to obtain $H_0$)

**Results**

We can observe that there is correlation approximal of $0.23$. With $p-\mathrm{value}\approx 0.001$. Due to size of that we can state that there is significant relation between distance of station and their measurements. To add interpretation to data i would need to have information about in what units distance is measured. 

In [ ]:
ozone = pl.read_csv("data/ozone.csv")
ozone = ozone.rename({'Lat': 'latitude', 'Lon': 'longitude', 'Av8top': 'ozone'})

coords = ozone.select(["longitude", "latitude"]).to_numpy()
values = ozone.select("ozone").to_numpy().ravel()

D_geo = squareform(pdist(coords))
D_oz = squareform(pdist(values[:, None]))

def mantel_test(D1, D2, n_perm=999):
    idx = np.triu_indices_from(D1, k=1)
    obs = spearmanr(D1[idx], D2[idx]).correlation
    perm = []
    for _ in range(n_perm):
        p = np.random.permutation(len(D1))
        perm.append(
            spearmanr(D1[p][:, p][idx], D2[idx]).correlation
        )
    p_value = np.mean(np.abs(perm) >= abs(obs))
    return obs, p_value

mantel_test(D_geo, D_oz)


### Exercise 3

I implemented divisive hierarchical clustering

The lowest misclassification rate was obtained for the agglomerative single linkage method, indicating that the data exhibit a continuous morphological structure. Cluster labels do not correspond directly to class labels; however, each true class is largely concentrated within a single dominant cluster. The complete and divisive methods yield higher error rates because they impose more restrictive assumptions regarding cluster compactness and homogeneity, which are not satisfied by the biological data.

In [ ]:
from sklearn.cluster import KMeans

def divisive_labels(X, n_clusters=5, random_state=0):
    """
    Simplify divisive hierarchical clustering
    """
    clusters = [np.arange(len(X))]

    while len(clusters) < n_clusters:
        # Highest variance claster
        ss = [
            np.sum((X[c] - X[c].mean(axis=0))**2)
            for c in clusters
        ]
        idx = np.argmax(ss)
        cluster = clusters.pop(idx)

        # Split claster by two using K-means
        kmeans = KMeans(
            n_clusters=2,
            n_init=10,
            random_state=random_state
        )
        labels = kmeans.fit_predict(X[cluster])

        clusters.append(cluster[labels == 0])
        clusters.append(cluster[labels == 1])

    # Switch clastters on label vector
    out = np.zeros(len(X), dtype=int)
    for i, c in enumerate(clusters):
        out[c] = i + 1  # labels 1, 2, ...

    return out


In [ ]:
scapulae = pl.read_csv("data/primate.scapulae.txt", separator=' ')
scapulae = scapulae.rename({'class': 'species'})

X = scapulae.select(pl.exclude("species")).to_numpy()
y = scapulae.select("species").to_numpy().ravel()

In [ ]:
scapulae = pl.read_csv("data/primate.scapulae.txt", separator=' ')
# scapulae = scapulae.rename({'class': 'species'}).drop_nans().drop_nulls()
scapulae = scapulae.rename({'class': 'species'})

X = scapulae.select(pl.exclude("species", "classdigit", "gamma", "genus"))
y = scapulae.select("classdigit")

X, y

In [ ]:
linkages = ["single", "complete", "average", "ward"]
results = {}

for method in linkages:
    Z = linkage(X, method=method)
    clusters = fcluster(Z, 5, criterion="maxclust")
    cm = confusion_matrix(y, clusters)
    results[method] = cm

# divisive hierarchical clustering
div_clusters = divisive_labels(X.to_numpy(), n_clusters=5)
cm_div = confusion_matrix(y, div_clusters)
results["divisive"] = cm_div


results

In [ ]:
def misclassification(cm):
    return 1 - np.sum(np.max(cm, axis=1)) / np.sum(cm)

rates = {m: misclassification(cm) for m, cm in results.items()}
rates


### Exercise 5

To obtain data we need firstly to project our 91x1 wektor into 14x14 matrix that allow ous to make analysis

Then we compare MDS metric (take distance into consideration) vs MDS no metric (it takes only order into consideration) 

In both cases we obtain circle structure. We can state that the structure is strong because we obtain same structure for distance and order. 

From those results we can state that human do not perceive colors in linear way but cyclic. Examined people are connecting very close highest frequency colors to lowest frequency, that is we do not discern them as frequency.   

In [ ]:
# -------------------------------------------------
# 1. Loading R data
# -------------------------------------------------
res = pyreadr.read_r("data/color.stimuli.rda")
x = list(res.values())[0]

# DataFrame -> numpy vector
x = x.to_numpy().ravel()

# -------------------------------------------------
# 2. Reproduce 14 x 14 
# -------------------------------------------------

n = 14
C = np.ones((n, n))

iu = np.triu_indices(n, k=1)

C[iu] = x
C[(iu[1], iu[0])] = x


np.fill_diagonal(C, 1.0)

# -------------------------------------------------
# 3. Change similarities to no similarities
# -------------------------------------------------
D = 1 - C

# -------------------------------------------------
# 4. Metric MDS
# -------------------------------------------------
mds_metric = MDS(
    n_components=2,
    dissimilarity="precomputed",
    metric=True,
    random_state=42
)

X_metric = mds_metric.fit_transform(D)

# -------------------------------------------------
# 5. No metric MDS
# -------------------------------------------------
mds_nonmetric = MDS(
    n_components=2,
    dissimilarity="precomputed",
    metric=False,
    random_state=42,
    n_init=20,
    max_iter=300
)

X_nonmetric = mds_nonmetric.fit_transform(D)

# -------------------------------------------------
# 6. Visualization
# -------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

labels = [str(i) for i in range(14)]

axes[0].scatter(X_metric[:, 0], X_metric[:, 1])

for i, label in enumerate(labels):
    axes[0].text(
        X_metric[i, 0],
        X_metric[i, 1],
        label,
        fontsize=14,
        ha="center",
        va="bottom"
    )

axes[0].set_title("Metric MDS")
axes[0].axis("equal")

axes[1].scatter(X_nonmetric[:, 0], X_nonmetric[:, 1])

for i, label in enumerate(labels):
    axes[1].text(
        X_nonmetric[i, 0],
        X_nonmetric[i, 1],
        label,
        fontsize=14,
        ha="center",
        va="bottom"
    )

axes[1].set_title("Non-metric MDS")
axes[1].axis("equal")


# ### OLD
# fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# axes[0].scatter(X_metric[:, 0], X_metric[:, 1])
# axes[0].set_title("Metric MDS")
# axes[0].axis("equal")

# axes[1].scatter(X_nonmetric[:, 0], X_nonmetric[:, 1])
# axes[1].set_title("Non-metric MDS")
# axes[1].axis("equal")

# plt.show()

### Exercise 6

1. 

For (m = 15), the OOB misclassification rate stabilizes and reaches its minimum, after which it starts to increase. For larger values of (m), too many features are considered at each split, which reduces randomness and leads to overly similar trees, making it harder to identify robust splitting patterns. For smaller values of (m), the level of randomness is too high, resulting in less stable models and larger variability of the OOB error across replications.

It should be noted that these fluctuations occur at very low error levels; therefore, the differences are relatively small and should not be overinterpreted, as they represent only minor deviations in overall performance.

With respect to parameter (B), increasing the number of trees does not lead to overfitting. Instead, it primarily affects the variability of the OOB error: for larger (B), the interquartile range is substantially smaller, indicating more stable and reliable error estimates compared to smaller forests.

2. 

The SOM yields a high average purity of approximately 0.96, indicating that most neurons are dominated by observations from a single class. This suggests a strong degree of topological separation between benign and malignant tumours in the feature space. However, since SOM is an unsupervised method, this result should not be interpreted as classification performance, but rather as evidence that the underlying data structure aligns well with the class labels.

In [ ]:
res = pyreadr.read_r("data/wdbc.rda")
df = list(res.values())[0]

X = df.drop(columns=["class"]).to_numpy()
y = df["class"].to_numpy()


In [ ]:
def rf_oob_experiment(X, y, m, B, n_rep=100):
    rates = []

    for seed in range(n_rep):
        rf = RandomForestClassifier(
            n_estimators=B,
            max_features=m,
            bootstrap=True,
            oob_score=True,
            random_state=seed,
            n_jobs=-1
        )

        rf.fit(X, y)
        rates.append(1 - rf.oob_score_)

    return np.array(rates)

In [ ]:
# experiment 1 to obtain optimal m value 
configs = [
    (2, 100),
    (5, 100),
    (10, 100),
    (15, 100),
    (20, 100),
]

results = {}

for m, B in configs:
    results[(m, B)] = rf_oob_experiment(X, y, m, B)


In [ ]:
# experiment 2 to obtain optimal b value 
m = 15
configs_2 = [
    (m, 10),
    (m, 20),
    (m, 40),
    (m, 60),
    (m, 80),
    (m, 100),
]

results_2 = {}

for m, B in configs_2:
    results_2[(m, B)] = rf_oob_experiment(X, y, m, B)

In [ ]:
plt.figure(figsize=(10, 5))

plt.boxplot(
    results.values(),
    labels=[f"m={m}, B={B}" for m, B in results.keys()]
)

plt.ylabel("OOB Misclassification Rate")
plt.title("Random Forest OOB Error (100 replications)")
plt.xticks(rotation=30)
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))

plt.boxplot(
    results_2.values(),
    labels=[f"m={m}, B={B}" for m, B in results_2.keys()]
)

plt.ylabel("OOB Misclassification Rate")
plt.title("Random Forest OOB Error (100 replications)")
plt.xticks(rotation=30)
plt.show()


minisom part 

In [ ]:
from minisom import MiniSom

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

som = MiniSom(
    x=10,
    y=10,
    input_len=X_scaled.shape[1],
    sigma=1.0,
    learning_rate=0.5,
    random_seed=42
)

som.random_weights_init(X_scaled)
som.train_batch(X_scaled, num_iteration=2000)


In [ ]:
# Assign observations to neurons, check compatibility with classes
from sklearn.preprocessing import LabelEncoder

# BMU for each observation
bmus = np.array([som.winner(x) for x in X_scaled])

# Encode class labels (B, M) -> (0, 1)
le = LabelEncoder()
y_num = le.fit_transform(y)

# Map BMUs to class labels
mapping = defaultdict(list)

for i, (x, y_) in enumerate(bmus):
    mapping[(x, y_)].append(y_num[i])

# Compute purity
purity = []

for node, labels in mapping.items():
    counts = np.bincount(labels)
    purity.append(np.max(counts) / np.sum(counts))

np.mean(purity)
